In [17]:
def test_min_eating_speed(solution):
    # Test case 1 (Example 1)
    piles = [1, 4, 3, 2]
    h = 9
    expected = 2
    print(f"test case: piles={piles}, h={h}")
    result = solution.minEatingSpeed(piles, h)
    assert result == expected, f"Test1 failed: got {result}, expected {expected}"

    # Test case 2 (Example 2)
    piles = [25, 10, 23, 4]
    h = 4
    expected = 25
    print(f"test case: piles={piles}, h={h}")
    result = solution.minEatingSpeed(piles, h)
    assert result == expected, f"Test2 failed: got {result}, expected {expected}"

    # Test case 3 (LeetCode Example)
    piles = [3, 6, 7, 11]
    h = 8
    expected = 4
    print(f"test case: piles={piles}, h={h}")
    result = solution.minEatingSpeed(piles, h)
    assert result == expected, f"Test3 failed: got {result}, expected {expected}"

In [36]:
import math
from typing import List

def check_sum(k, h, piles): #indicator of if k threshold is large enough such that equation holds.
    if h >= sum([math.ceil(piles[i]/ k) for i in range(len(piles))]):
        return True
    else:
        return False
        
def helper(piles, h, k, has_passed, largest):
    #base case if we reach no further bisection possible  where 
    if k == 1 or k == largest: #edges
        return k
    
    if not has_passed: #keep going right until true
        k_right = k * 3/2
        return helper(piles, h, k_right, check_sum(k_right, h, piles), largest) #bisect right
    else: #keep going left until false
        k_left = k * 1/2
        left_sum_passed = check_sum(k_left, h, piles)
        if not left_sum_passed: #first false on the left 
            return k 
        else: return helper(piles, h, k_left, left_sum_passed, largest) #bisect left
        

class Solution:


    def minEatingSpeed(self, piles: List[int], h: int) -> int:
        # upper bound for time to take is k = 1 and all piles at max size z
        # hence z * h = m (maximum possible k value), then we can binary search [1, z * h]
        m = 2**math.ceil(math.log2(max(piles) * h)) # O(n) #find the nearest 2^n s.t. 2^n >= m*h 
        k_start = m/2
        print(f"k_start: {k_start}, piles: {piles}, h: {h}")
        return helper(piles, h, k_start, check_sum(k_start, h, piles), m)
        

In [37]:
solution = Solution()
test_min_eating_speed(solution)


test case: piles=[1, 4, 3, 2], h=9
k_start: 32.0, piles: [1, 4, 3, 2], h: 9
test case: piles=[25, 10, 23, 4], h=4
k_start: 64.0, piles: [25, 10, 23, 4], h: 4


AssertionError: Test2 failed: got 32.0, expected 25

### Critique Hint (Edge Case)
Your search variable `k` is using `/`, `* 3/2`, and `* 1/2`, so it becomes a **float**. That causes two problems:

1. Your stopping condition relies on exact equality (`k == 1` or `k == max(piles) * h`), which is fragile once `k` is fractional.
2. The answer must be an **integer speed**, but this search can skip valid integer candidates (like `25`) and land on `32`.

Think about the edge case `h == len(piles)`: Koko has exactly one hour per pile, so the minimum valid speed must be `max(piles)`. Use that as a sanity check for your bounds and search updates.

Non-spoiler direction: keep `k` integral throughout and do monotonic binary search on `[1, max(piles)]`.


In [ ]:
import math
from typing import List

def check_sum(k, h, piles): #indicator of if k threshold is large enough such that equation holds.
    if h >= sum([math.ceil(piles[i]/ k) for i in range(len(piles))]):
        return True
    else:
        return False
        
def helper(piles, h, k, has_passed, largest):
    #base case if we reach no further bisection possible  where 
    if not has_passed: #keep going right until true
        print(f"going right: k: {k}, has_passed: {has_passed}")
        k_right = k * 3/2
        return helper(piles, h, k_right, check_sum(k_right, h, piles), largest) #bisect right
    else: #keep going left until false
        k_left = k * 1/2
        left_sum_passed = check_sum(k_left, h, piles)
        print(f"going left: k: {k}  has_passed: {has_passed}, k_left: {k_left}, left_sum_passed: {left_sum_passed}")

        if not left_sum_passed: #first true on the left 
            return 
        else: return helper(piles, h, k_left, left_sum_passed, largest) #bisect left
        

class Solution2:


    def minEatingSpeed(self, piles: List[int], h: int) -> int:
        # upper bound for time to take is k = 1 and all piles at max size z
        # hence z * h = m (maximum possible k value), then we can binary search [1, z * h]
        m = 2**math.ceil(math.log2(max(piles) )) # O(n) #find the nearest 2^n s.t. 2^n >= m*h 
        k_start = m/2
        print(f"k_start: {k_start}, piles: {piles}, h: {h}")
        return helper(piles, h, k_start, check_sum(k_start, h, piles), m)
        

In [57]:
solution = Solution2()
test_min_eating_speed(solution)


test case: piles=[1, 4, 3, 2], h=9
k_start: 2.0, piles: [1, 4, 3, 2], h: 9
going left: k: 2.0  has_passed: True, k_left: 1.0, left_sum_passed: False
test case: piles=[25, 10, 23, 4], h=4
k_start: 16.0, piles: [25, 10, 23, 4], h: 4
going right: k: 16.0, has_passed: False
going right: k: 24.0, has_passed: False
going left: k: 36.0  has_passed: True, k_left: 18.0, left_sum_passed: False


AssertionError: Test2 failed: got 36.0, expected 25

### Critique Hint (Solution2)
Current behavior: your function returns `None` for test cases.

Why this happens in your current state:
- In `helper`, when `left_sum_passed` is `False`, you do bare `return` (no value), so recursion unwinds as `None`.
- There is no explicit convergence/base condition for "search interval is finished", so correctness depends on hitting that bare return path.

Edge-case check to drive your fix:
- `piles = [30,11,23,4,20], h = 5`
- Expected answer is `30` (one hour per pile, so speed must be at least max pile).

Non-spoiler direction: make every branch return an integer `k`, and use a clear integer binary-search stop condition (`left <= right` or equivalent bounds crossing).


### Binary Search (Non-Spoiler, General)
Use binary search when:
- You are looking for a minimum/maximum valid value.
- A yes/no check is monotonic (once true, it stays true; or once false, it stays false).

For Koko, think of a candidate speed `k` and a check function:
- `feasible(k)`: can she finish all piles within `h` hours?
- If a speed is feasible, any faster speed is also feasible.
- If a speed is not feasible, any slower speed is also not feasible.

That monotonic shape is why binary search works.

General pattern:
1. Choose integer bounds `[left, right]` that definitely contain the answer.
2. While the search window is valid (`left <= right`):
   - Pick midpoint `mid` as an integer.
   - Run `feasible(mid)`.
   - If feasible, record `mid` as a candidate answer and move left (`right = mid - 1`) to try smaller valid values.
   - If not feasible, move right (`left = mid + 1`) to find a large-enough value.
3. Return the best recorded feasible candidate.

Important invariants to protect:
- Bounds and midpoint stay integers.
- Every loop iteration strictly shrinks the interval.
- Every branch returns/updates consistently (no implicit `None` path).

Quick sanity anchors (for this problem type):
- Lower bound is at least `1`.
- Upper bound can be `max(piles)` (always feasible in one hour per pile).
- If `h == len(piles)`, answer must be `max(piles)`.
